In [5]:
import pandas as pd
import numpy as np

interests = ['Data Science','Web Dev','Networking','Cybersecurity','AI/ML','Software Eng','Database','Mobile Dev']

np.random.seed(42)
n = 200

semesters = np.random.randint(1, 9, n)
cgpa = np.round(np.random.uniform(1.0, 4.0, n), 2)
attendance = np.random.randint(40, 101, n)
failed = [0 if s <= 2 else np.random.randint(0, 5) for s in semesters]
credits = np.random.choice([12, 15, 18, 21], n)
interest = np.random.choice(interests, n)
assignments = np.round(np.random.uniform(40, 100, n), 1)
mid_marks = np.random.randint(20, 51, n)
final_marks = np.random.randint(30, 51, n)

df = pd.DataFrame({
    'student_id': [f'STU-{str(i).zfill(4)}' for i in range(1, n+1)],
    'semester': semesters,
    'cgpa': cgpa,
    'attendance_pct': attendance,
    'failed_subjects': failed,
    'credit_hours': credits,
    'interest_area': interest,
    'assignments_avg': assignments,
    'mid_marks': mid_marks,
    'final_marks': final_marks
})

print(df.shape)
df.head()

(200, 10)


,student_id,semester,cgpa,attendance_pct,failed_subjects,credit_hours,interest_area,assignments_avg,mid_marks,final_marks
0,STU-0001,7,1.09,84,3,21,Data Science,47.5,41,38
1,STU-0002,4,2.91,71,3,18,Mobile Dev,83.9,31,48
2,STU-0003,5,1.94,91,4,12,Data Science,96.3,43,45
3,STU-0004,7,2.53,69,0,18,Networking,50.9,36,37
4,STU-0005,3,3.72,86,2,12,AI/ML,44.0,29,47


In [6]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   student_id       200 non-null    object 
 1   semester         200 non-null    int64  
 2   cgpa             200 non-null    float64
 3   attendance_pct   200 non-null    int64  
 4   failed_subjects  200 non-null    int64  
 5   credit_hours     200 non-null    int64  
 6   interest_area    200 non-null    object 
 7   assignments_avg  200 non-null    float64
 8   mid_marks        200 non-null    int64  
 9   final_marks      200 non-null    int64  
dtypes: float64(2), int64(6), object(2)
memory usage: 15.8+ KB


,semester,cgpa,attendance_pct,failed_subjects,credit_hours,assignments_avg,mid_marks,final_marks
count,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000
mean,4.560000,2.523000,71.315000,1.670000,15.975000,69.072000,34.910000,39.575000
std,2.283082,0.878246,17.002979,1.585455,3.238334,17.096011,8.378718,5.689131
min,1.000000,1.020000,40.000000,0.000000,12.000000,40.300000,20.000000,30.000000
25%,3.000000,1.745000,58.000000,0.000000,12.000000,53.450000,28.000000,34.000000
50%,4.000000,2.565000,71.000000,1.000000,15.000000,67.750000,36.000000,40.000000
75%,7.000000,3.287500,87.000000,3.000000,18.000000,83.975000,42.000000,44.000000
max,8.000000,3.970000,100.000000,4.000000,21.000000,99.800000,50.000000,50.000000


In [7]:
def assign_labels(row):
    # student level
    if row['cgpa'] >= 2.5 and row['attendance_pct'] >= 70 and row['failed_subjects'] <= 1:
        level = 'Upper'
    else:
        level = 'Lower'

    # risk level
    if row['cgpa'] >= 3.0 and row['attendance_pct'] >= 75 and row['failed_subjects'] == 0:
        risk = 'Low'
    elif row['cgpa'] >= 2.0 and row['attendance_pct'] >= 60 and row['failed_subjects'] <= 2:
        risk = 'Medium'
    else:
        risk = 'High'

    return pd.Series([level, risk])

df[['student_level', 'risk_level']] = df.apply(assign_labels, axis=1)

print(df['student_level'].value_counts())
print()
print(df['risk_level'].value_counts())

student_level
Lower    175
Upper     25
Name: count, dtype: int64

risk_level
High      142
Medium     45
Low        13
Name: count, dtype: int64


In [8]:
from sklearn.utils import resample

# separate classes
lower = df[df['student_level'] == 'Lower']
upper = df[df['student_level'] == 'Upper']

# upsample minority class
upper_upsampled = resample(upper, replace=True, n_samples=len(lower), random_state=42)
df_balanced = pd.concat([lower, upper_upsampled])

print(df_balanced['student_level'].value_counts())
print(df_balanced['risk_level'].value_counts())

student_level
Lower    175
Upper    175
Name: count, dtype: int64
risk_level
High      142
Medium    113
Low        95
Name: count, dtype: int64


In [9]:
from sklearn.preprocessing import LabelEncoder

features = df_balanced.drop(columns=['student_id', 'interest_area', 'student_level', 'risk_level'])

le_level = LabelEncoder()
le_risk = LabelEncoder()

y_level = le_level.fit_transform(df_balanced['student_level'])
y_risk = le_risk.fit_transform(df_balanced['risk_level'])

print("Level classes:", le_level.classes_)
print("Risk classes:", le_risk.classes_)
print("Features shape:", features.shape)

Level classes: ['Lower' 'Upper']
Risk classes: ['High' 'Low' 'Medium']
Features shape: (350, 8)


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# --- Level Model ---
X_train, X_test, y_train, y_test = train_test_split(features, y_level, test_size=0.2, random_state=42)
model_level = RandomForestClassifier(n_estimators=100, random_state=42)
model_level.fit(X_train, y_train)
print("=== Student Level Model ===")
print(classification_report(y_test, model_level.predict(X_test), target_names=le_level.classes_))

# --- Risk Model ---
X_train, X_test, y_train, y_test = train_test_split(features, y_risk, test_size=0.2, random_state=42)
model_risk = RandomForestClassifier(n_estimators=100, random_state=42)
model_risk.fit(X_train, y_train)
print("=== Risk Level Model ===")
print(classification_report(y_test, model_risk.predict(X_test), target_names=le_risk.classes_))

=== Student Level Model ===
              precision    recall  f1-score   support

       Lower       1.00      1.00      1.00        37
       Upper       1.00      1.00      1.00        33

    accuracy                           1.00        70
   macro avg       1.00      1.00      1.00        70
weighted avg       1.00      1.00      1.00        70

=== Risk Level Model ===
              precision    recall  f1-score   support

        High       0.97      1.00      0.98        32
         Low       1.00      1.00      1.00        20
      Medium       1.00      0.94      0.97        18

    accuracy                           0.99        70
   macro avg       0.99      0.98      0.99        70
weighted avg       0.99      0.99      0.99        70



In [11]:
import joblib
joblib.dump(model_level, 'model_level.pkl')
joblib.dump(model_risk, 'model_risk.pkl')
joblib.dump(le_level, 'le_level.pkl')
joblib.dump(le_risk, 'le_risk.pkl')
print("Models saved!")

Models saved!
